# N3 · Where does assimilation improve, and where does it degrade?

Four questions, in order:

1. **How often does the analysis beat the prior**, and does the answer depend on
   whether we score at the observation point, over the localization-weighted
   volume, or over the whole cutoff zone?
2. **Where** — in space and in height — does degradation concentrate?
3. **Does the metric matter?** RMSE only sees the ensemble mean. CRPS scores the
   whole distribution, so it also charges for a spread collapse. Do they ever
   disagree about the same point?
4. **Do prior conditions predict degradation?** If they do, an extra QC could
   refuse those observations before they are ever assimilated.

### The sign convention

Everything plotted as *skill* is `prior − analysis`, so **positive always means
the analysis is better**. `nbcommon` enforces this — it exports no other
difference, and `assert_convention()` fails at import if the sign is ever
flipped. See `nbcommon.CONVENTION`, printed below.

### Reductions

| suffix | meaning |
|---|---|
| `_point_` | at the observation grid point — the most optimistic view |
| `_w_` | mean over the subdomain weighted by the localization weight ρ |
| `_u_` | unweighted mean over the whole cutoff zone (ρ > 0) — the most pessimistic |

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import nbcommon as nb
nb.banner()
nb.assert_convention()

# ── configuration (N4 repeats these literals verbatim) ─────────────────────
# Point these at the two sweep output directories under data/ once the runs land.
# `tag` is the directory name; `loc` is the isotropic localization scale it used.
# One config per scale -- loc_x/loc_y/loc_z are a Cartesian product in
# _build_combos, so a single config listing both would give 8 mixed-anisotropy
# combos. N4 must repeat this literal verbatim; the cross-check verifies it.
RUNS = [dict(tag="WS_sweep_L0.1", loc=0.1, hour="18"),
        dict(tag="WS_sweep_L2.0", loc=2.0, hour="18")]

METRICS    = ("rmse", "crps")
REF_METHOD = ("TEnKF", 2)          # the combo N3's headline table is about
N_MIN      = 8                     # candidate QC: minimum members with signal
NI_MAX     = 2.5                   # candidate QC: maximum normalised innovation

PRED_COLS = ["dep_b", "n_active_f_point", "spread_f_point_obs",
             "skew_f_point_obs", "kurt_f_point_obs"]

In [ ]:
COLS = nb.columns_for(metrics=METRICS, vars=("obs",), extra=PRED_COLS)
df   = nb.load_runs(RUNS, columns=COLS)

R       = nb.obs_error_var(RUNS[0]["tag"])
DBZ_MIN = nb.dbz_min_of(RUNS[0]["tag"])
for r in RUNS[1:]:
    assert np.isclose(nb.obs_error_var(r["tag"]), R), "runs disagree on obs_error_var"
    assert np.isclose(nb.dbz_min_of(r["tag"]), DBZ_MIN), "runs disagree on dbz_min"
print(f"obs_error_var R = {R}   dbz_min = {DBZ_MIN}")

COMBOS = nb.combos_in(df)
A = nb.align(df, COMBOS)
print(A)
print(A.coverage().to_string(index=False))

# The two localization runs must cover the same observations, or every
# loc-to-loc comparison below is confounded by a different point set.
raw_sets = {}
for r in RUNS:
    sub = df[df.run == r["tag"]]
    raw_sets[r["tag"]] = set(nb.point_key(sub).unique())
sizes = {k: len(v) for k, v in raw_sets.items()}
shared = len(set.intersection(*raw_sets.values()))
print(f"\nobserved points per run: {sizes}")
print(f"shared by every run    : {shared:,}")
assert shared > 0, "the runs share no observation points"
if len(set(sizes.values())) > 1:
    print("WARNING: the runs observed different point sets; align() has already "
          "restricted everything below to the intersection.")

print(f"\ncommon points across all {len(A.combos)} combos: {A.n_points:,} "
      f"({A.dropped:,} dropped by the intersection)")

## 1 · The headline

`frac_improved` is the fraction of observation points where the analysis beat the
prior by more than the physical tolerance `TOL[var]`; points inside the tolerance
are ties and count for neither side. The bars carry Wilson 95 % intervals — with
this many points they are narrow, which is the point: the *uncertainty* on these
fractions is negligible compared with how much they move between configurations.

In [ ]:
HEAD = pd.concat([nb.skill_summary(A, "rmse", "obs", red) for red in nb.REDUCTIONS],
                 ignore_index=True)
show = HEAD[(HEAD.method == REF_METHOD[0]) & (HEAD.ntemp == REF_METHOD[1])]
print(show[["loc_km", "red", "n_points", "n_nan", "median_skill",
            "frac_improved", "frac_degraded", "ci_lo", "ci_hi"]].to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(nb.PAGE_W, nb.PAGE_W * 0.34), sharey=True)
locs = sorted(HEAD.loc_km.unique())
for ax, metric_col, lab in zip(axes, ["frac_improved", "median_skill"],
                               ["fraction of points improved",
                                nb.skill_label("rmse", "obs", "w")]):
    w = 0.26
    for t, red in enumerate(nb.REDUCTIONS):
        s = show[show.red == red].sort_values("loc_km")
        x = np.arange(len(locs)) + (t - 1) * w
        yerr = None
        if metric_col == "frac_improved":
            yerr = np.vstack([s[metric_col] - s.ci_lo, s.ci_hi - s[metric_col]])
        ax.bar(x, s[metric_col], w, yerr=yerr, capsize=2,
               label=nb.RED_LABELS[red],
               color=[nb.C_NONE, nb.C_NT2, nb.C_NT1][t])
    ax.set_xticks(np.arange(len(locs)))
    ax.set_xticklabels([nb.loc_label(l) for l in locs])
    ax.set_ylabel(lab if metric_col == "frac_improved" else "")
    ax.set_title(lab, fontsize=8)
    if metric_col == "frac_improved":
        ax.axhline(0.5, color="k", lw=0.8, ls="--")
    else:
        ax.axhline(0.0, color="k", lw=0.8)
axes[0].legend(fontsize=7)
fig.suptitle(f"{nb.combo_label(*REF_METHOD)} — RMSE skill in observation space", y=1.02)
nb.save_fig(fig, "N3_fig1_headline")
plt.show()

**How to read this.** `_point_` scores only the assimilated cell, where the
increment is largest and best constrained, so it flatters the filter. `_u_`
averages over every cell the localization touched, including those far from the
observation where the increment is mostly sampling noise — it is the honest
number if you care about the analysis as a field.

At **L = 0.1 km the three reductions nearly collapse**, because the cutoff zone
is only a 3×3×3 box: there is barely any "volume" to average over. The spread
between reductions at L = 2.0 km is the real content of this figure.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(nb.PAGE_W, nb.PAGE_W * 0.52),
                         sharex=True, sharey=True)
for r, loc in enumerate(locs):
    for c, red in enumerate(nb.REDUCTIONS):
        ax = axes[r, c]
        combo = [k for k in A.combos if k[0] == REF_METHOD[0]
                 and int(k[1]) == REF_METHOD[1] and np.isclose(float(k[3]), loc)][0]
        s = nb.skill(A.frames[combo], "rmse", "obs", red).to_numpy(float)
        s = s[np.isfinite(s)]
        ax.hist(s, bins=120, color=nb.C_NT2, alpha=0.85)
        ax.set_xscale("symlog", linthresh=0.1)
        ax.axvline(0, color="k", lw=1.0)
        pos, neg = 100 * (s > nb.TOL["obs"]).mean(), 100 * (s < -nb.TOL["obs"]).mean()
        ax.text(0.03, 0.92, f"better {pos:.0f} %", transform=ax.transAxes,
                fontsize=7, color=nb.C_NT2, va="top")
        ax.text(0.97, 0.92, f"worse {neg:.0f} %", transform=ax.transAxes,
                fontsize=7, color="#e34948", ha="right", va="top")
        if r == 0: ax.set_title(nb.RED_LABELS[red], fontsize=8)
        if c == 0: ax.set_ylabel(f"{nb.loc_label(loc)}\ncount")
        if r == 1: ax.set_xlabel(nb.skill_label("rmse", "obs", red).split("[")[0])
fig.suptitle("Distribution of per-point RMSE skill (symlog x; right of 0 = improved)",
             y=1.01)
nb.save_fig(fig, "N3_fig2_skill_distributions")
plt.show()

## 2 · Where degradation lives

Two views of the same thing: horizontally, and against height. The height profile
matters because at `L = 0.1 km` the vertical localization is only 0.1 km, so
level `k` is essentially coupled to itself — the low levels behave differently
from the rest for a structural reason, not a meteorological one.

In [ ]:
# coarse_map_km needs a (nx, ny, nz, 3) pos_km to convert bin indices to km. The
# sweep rows already carry x_km/y_km, so reconstruct the axes from them rather than
# reading a multi-GB subset back in.
NXG = int(df["i"].max()) + 1
NYG = int(df["j"].max()) + 1
DX = float(df["x_km"].iloc[1] - df["x_km"].iloc[0]) / max(int(df["i"].iloc[1] - df["i"].iloc[0]), 1)
DY = DX
POS_KM = np.zeros((NXG, NYG, 1, 3), np.float32)
POS_KM[..., 0] = (np.arange(NXG) * abs(DX))[:, None, None]
POS_KM[..., 1] = (np.arange(NYG) * abs(DY))[None, :, None]
print(f"map grid {NXG} x {NYG}, dx = {abs(DX):.2f} km")

fig, axes = plt.subplots(1, 3, figsize=(nb.PAGE_W, nb.PAGE_W * 0.30))
for ax, loc in zip(axes[:2], locs):
    combo = [k for k in A.combos if k[0] == REF_METHOD[0]
             and int(k[1]) == REF_METHOD[1] and np.isclose(float(k[3]), loc)][0]
    f = A.frames[combo].copy()
    f["_sk"] = nb.skill(f, "rmse", "obs", "w")
    xs, ys, grid = nb.coarse_map_km(f, "_sk", POS_KM, stride=20, agg="mean")
    v = np.nanpercentile(np.abs(grid), 98) or 1.0
    pm = ax.pcolormesh(xs, ys, grid.T, cmap=nb.CMAP_SKILL,
                       vmin=-v, vmax=v, shading="auto")
    plt.colorbar(pm, ax=ax, shrink=0.8)
    ax.set_title(nb.loc_label(loc), fontsize=8)
    ax.set_xlabel("x [km]"); ax.set_ylabel("y [km]")
    ax.grid(False)

ax = axes[2]
for loc in locs:
    combo = [k for k in A.combos if k[0] == REF_METHOD[0]
             and int(k[1]) == REF_METHOD[1] and np.isclose(float(k[3]), loc)][0]
    f = A.frames[combo]
    s = nb.skill(f, "rmse", "obs", "w")
    prof = pd.DataFrame({"k": f["k"].to_numpy(), "s": s.to_numpy()}).groupby("k")["s"]
    ax.plot(prof.median(), prof.median().index, "-o", ms=3,
            color=nb.loc_color(loc), label=nb.loc_label(loc))
ax.axvline(0, color="k", lw=0.8)
ax.set_xlabel("median RMSE skill [dBZ]  (>0 better)")
ax.set_ylabel("model level k")
ax.legend(fontsize=7)
fig.suptitle(f"{nb.combo_label(*REF_METHOD)} — where the analysis helps and hurts", y=1.03)
nb.save_fig(fig, "N3_fig3_where")
plt.show()

## 3 · RMSE and CRPS — do they agree?

RMSE compares the ensemble *mean* to the truth. CRPS compares the whole predictive
distribution, so an analysis that moves the mean the right way while collapsing
the spread can improve RMSE and worsen CRPS at the same point.

If the two metrics disagree on a material fraction of points, then "the
assimilation improved the analysis" is not a well-posed statement without saying
which metric was used — and that is exactly the kind of ambiguity that lets two
analyses of the same data reach opposite conclusions.

In [ ]:
from scipy.stats import spearmanr

fig, axes = plt.subplots(1, 2, figsize=(nb.PAGE_W, nb.PAGE_W * 0.36))
for ax, loc in zip(axes, locs):
    combo = [k for k in A.combos if k[0] == REF_METHOD[0]
             and int(k[1]) == REF_METHOD[1] and np.isclose(float(k[3]), loc)][0]
    f = A.frames[combo]
    sr = nb.skill(f, "rmse", "obs", "w").to_numpy(float)
    sc = nb.skill(f, "crps", "obs", "w").to_numpy(float)
    m  = np.isfinite(sr) & np.isfinite(sc)
    sr, sc = sr[m], sc[m]

    rho = spearmanr(sr, sc).correlation
    disagree = np.mean(np.sign(sr) != np.sign(sc))

    xl, yl = nb.lims(sr, 1, 99), nb.lims(sc, 1, 99)
    hb = ax.hexbin(sr, sc, gridsize=55, bins="log", cmap=nb.CMAP_SEQ,
                   extent=(xl[0], xl[1], yl[0], yl[1]), mincnt=1, linewidths=0)
    ax.axhline(0, color="k", lw=0.7); ax.axvline(0, color="k", lw=0.7)
    lo = max(xl[0], yl[0]); hi = min(xl[1], yl[1])
    ax.plot([lo, hi], [lo, hi], "--", color="#888", lw=0.8)
    ax.set_xlabel("RMSE skill [dBZ]"); ax.set_ylabel("CRPS skill [dBZ]")
    ax.set_title(f"{nb.loc_label(loc)}   Spearman $\\rho$ = {rho:.3f}\n"
                 f"sign disagreement: {100*disagree:.1f} % of points", fontsize=8)
    plt.colorbar(hb, ax=ax, shrink=0.8, label="points")
    ax.grid(False)
fig.suptitle("Do the two metrics rank the same point the same way?", y=1.04)
nb.save_fig(fig, "N3_fig4_rmse_vs_crps")
plt.show()

print("Quadrant II/IV = the two metrics disagree about whether that point improved.")
print("Off-diagonal mass below the 1:1 line = CRPS is stricter, i.e. the analysis")
print("bought its mean improvement partly with a spread collapse.")

## 4 · Do prior conditions predict degradation?

Every predictor here is a **stored sweep column** — nothing recomputes H(x). After
the reflectivity-metrics change the sweep already carries members-with-signal,
prior spread, skewness and kurtosis in observation space, which is what made the
old `build_obs_predictors.py` redundant.

| predictor | definition |
|---|---|
| `n_active_f_point` | members with `H(x) > dbz_min`, of `Ne` |
| `frac_floor` | `1 − n_active/Ne` — the share of members sitting at the clear-air floor |
| `spread_f_point_obs` | prior ensemble spread σ_H [dBZ] |
| `norm_innov` | `\|d\| / sqrt(σ_H² + R)` — innovation in units of what the filter expects |
| `abs_skew_f`, `kurt_f_point_obs` | prior non-Gaussianity |

In [ ]:
PRED = {}
for loc in locs:
    combo = [k for k in A.combos if k[0] == REF_METHOD[0]
             and int(k[1]) == REF_METHOD[1] and np.isclose(float(k[3]), loc)][0]
    f = nb.add_predictors(A.frames[combo], R=R, Ne=int(A.frames[combo]["Ne"].iloc[0]))
    f["skill_rmse"] = nb.skill(f, "rmse", "obs", "w")
    f["skill_crps"] = nb.skill(f, "crps", "obs", "w")
    f["degraded"]   = f["skill_rmse"] < -nb.TOL["obs"]
    PRED[loc] = f

summary = pd.DataFrame({
    p: PRED[locs[0]][p].describe()[["min", "25%", "50%", "75%", "max"]]
    for p, _, _ in nb.PREDICTORS if p in PRED[locs[0]]}).T
print(f"predictor ranges at {nb.loc_label(locs[0])}:\n")
print(summary.round(3).to_string())
print(f"\ndegraded fraction: " +
      "   ".join(f"{nb.loc_label(l)} {100*PRED[l]['degraded'].mean():.1f} %" for l in locs))

In [ ]:
PRED_LABEL = {p: lab for p, lab, _ in nb.PREDICTORS}
use = [(p, lab, clip) for p, lab, clip in nb.PREDICTORS if p in PRED[locs[0]]]
fig, axes = plt.subplots(2, 3, figsize=(nb.PAGE_W, nb.PAGE_W * 0.48))
for ax, (p, lab, clip) in zip(axes.ravel(), use[:6]):
    f = PRED[locs[0]]
    v = f[p].to_numpy(float)
    lo, hi = np.percentile(v[np.isfinite(v)], clip)
    bins = np.linspace(lo, hi, 45)
    ax.hist(v[~f["degraded"]], bins=bins, density=True, histtype="step",
            lw=1.3, color=nb.C_NT2, label="improved")
    ax.hist(v[f["degraded"]], bins=bins, density=True, histtype="step",
            lw=1.3, color="#e34948", label="degraded")
    ax.set_xlabel(lab, fontsize=7); ax.set_ylabel("density", fontsize=7)
axes.ravel()[0].legend(fontsize=7)
fig.suptitle(f"Prior conditions, conditioned on the outcome — "
             f"{nb.combo_label(*REF_METHOD)}, {nb.loc_label(locs[0])}", y=1.02)
nb.save_fig(fig, "N3_fig5_predictor_distributions")
plt.show()

In [ ]:
PLANES = [("n_active_f_point", "norm_innov"),
          ("spread_f_point_obs", "abs_dep_b"),
          ("frac_floor", "norm_innov")]

fig, axes = plt.subplots(len(locs), len(PLANES),
                         figsize=(nb.PAGE_W, nb.PAGE_W * 0.55))
for r, loc in enumerate(locs):
    f = PRED[loc]
    for c, (px, py) in enumerate(PLANES):
        ax = axes[r, c]
        xl, yl = nb.lims(f[px], 0, 99.5), nb.lims(f[py], 0, 99.5)
        hb = nb.prob_hexbin(ax, f[px].to_numpy(float), f[py].to_numpy(float),
                            f["degraded"].to_numpy(), xl, yl, gridsize=40, mincnt=25)
        ax.set_xlabel(PRED_LABEL.get(px, px), fontsize=7)
        ax.set_ylabel(PRED_LABEL.get(py, py), fontsize=7)
        if c == 0:
            ax.text(-0.32, 0.5, nb.loc_label(loc), transform=ax.transAxes,
                    rotation=90, va="center", ha="center", fontweight="bold")
        ax.grid(False)
fig.colorbar(hb, ax=axes, shrink=0.6, pad=0.02, label="P(degraded)")
fig.suptitle("Probability that the analysis is worse than the prior, "
             "as a function of prior conditions", y=0.98)
nb.save_fig(fig, "N3_fig6_prob_degraded")
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(nb.PAGE_W, nb.PAGE_W * 0.48))
for ax, (p, lab, clip) in zip(axes.ravel(), use[:6]):
    for loc in locs:
        f = PRED[loc]
        v = f[p].to_numpy(float)
        lo, hi = np.percentile(v[np.isfinite(v)], clip)
        edges = np.linspace(lo, hi, 22)
        for metric, ls in (("skill_rmse", "-"), ("skill_crps", "--")):
            cen, mean, _, n = nb.binned_stats(v, f[metric].to_numpy(float),
                                              edges, min_count=40)
            ax.plot(cen, mean, ls, color=nb.loc_color(loc), lw=1.2,
                    label=f"{nb.loc_label(loc)} {'RMSE' if ls=='-' else 'CRPS'}")
    ax.axhline(0, color="k", lw=0.8)
    ax.set_xlabel(lab, fontsize=7)
    ax.set_ylabel("mean skill [dBZ]", fontsize=7)
axes.ravel()[0].legend(fontsize=5.5, ncol=2)
fig.suptitle("Mean skill against each prior predictor "
             "(solid RMSE, dashed CRPS; >0 = analysis better)", y=1.02)
nb.save_fig(fig, "N3_fig7_skill_vs_predictor")
plt.show()

## 5 · A candidate extra-QC rule — **DIAGNOSTIC ONLY**

> **Read this before quoting any number below.**
>
> Nothing here is implemented and nothing is re-run. The numbers are properties of
> the observations **that were actually assimilated in this experiment**. They say
> "among the points this rule would have kept, the analysis did such-and-such" —
> which is *not* the same as "an experiment run under this rule would score
> such-and-such".
>
> The two differ for a concrete reason: in multi-obs assimilation, removing an
> observation changes the analysis at its neighbours too. Only a re-run with the
> rule active in `_qc_pass` can measure that. Treat what follows as evidence that
> a rule is *worth testing*, not as its result.

In [ ]:
print(f"candidate rule:  n_active_f_point >= {N_MIN}  AND  norm_innov <= {NI_MAX}\n")
for loc in locs:
    f = PRED[loc]
    keep = (f["n_active_f_point"] >= N_MIN) & (f["norm_innov"] <= NI_MAX)
    ct = pd.crosstab(np.where(keep, "rule keeps", "rule drops"),
                     np.where(f["degraded"], "degraded", "improved"))
    print(f"── {nb.loc_label(loc)} ─────────────────────────────")
    print(ct.to_string())
    print(f"   retained            : {100*keep.mean():5.1f} % of observations")
    print(f"   P(degraded | kept)  : {100*f.loc[keep, 'degraded'].mean():5.1f} %")
    print(f"   P(degraded | dropped): {100*f.loc[~keep, 'degraded'].mean():5.1f} %")
    print(f"   median skill of the KEPT points: "
          f"{f.loc[keep, 'skill_rmse'].median():+.3f} dBZ  "
          f"(NOT the skill of a re-run under the rule)\n")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(nb.PAGE_W, nb.PAGE_W * 0.36))
for ax, loc in zip(axes, locs):
    f = PRED[loc]
    for ni, col in zip([1.5, 2.5, 4.0, np.inf],
                       [nb.C_NT2, nb.C_NONE, nb.C_NT1, "#888888"]):
        xs, ys = [], []
        for nmin in range(0, 41, 2):
            keep = (f["n_active_f_point"] >= nmin) & (f["norm_innov"] <= ni)
            if keep.sum() < 200:
                continue
            xs.append(100 * keep.mean())
            ys.append(100 * (1 - f.loc[keep, "degraded"].mean()))
        ax.plot(xs, ys, "-o", ms=2.5, color=col, lw=1.1,
                label=f"norm_innov $\\leq$ {ni:g}")
    keep = (f["n_active_f_point"] >= N_MIN) & (f["norm_innov"] <= NI_MAX)
    ax.plot(100 * keep.mean(), 100 * (1 - f.loc[keep, "degraded"].mean()),
            "*", ms=15, color="#111", zorder=5, label="chosen operating point")
    ax.axhline(100 * (1 - f["degraded"].mean()), color="k", ls=":", lw=1.0,
               label="no extra QC")
    ax.set_xlabel("observations retained [%]")
    ax.set_ylabel("share of retained points that improved [%]")
    ax.set_title(nb.loc_label(loc), fontsize=8)
axes[0].legend(fontsize=6.5)
fig.suptitle("Coverage curve: how selective must the rule be to raise the hit rate? "
             "(diagnostic, not a re-run)", y=1.03)
nb.save_fig(fig, "N3_fig8_qc_coverage")
plt.show()

## 6 · Handoff to N4

The digest below is what N4 pastes into its cross-check cell. If N4 ever computes
a different headline table from the same runs, its cell 5 raises `ConsistencyError`
and prints a row-aligned diff, so the two notebooks cannot quietly disagree.

In [ ]:
N3_HEADLINE = nb.skill_summary(A, "rmse", "obs", "w")
print(N3_HEADLINE.to_string(index=False))

digest = nb.publish("N3_headline", N3_HEADLINE)
print(f"\n\nPaste into N4 cell 2:\n\n    N3_DIGEST = \"{digest}\"\n")